# 01 — Data Collection

**Route:** C81 (Campbell Airport, Grayslake, IL) → KDLH (Duluth International, MN)

This is the first module in the VFR waypoint recommender project. The goal
here is purely `pandas`/data-wrangling: pull real candidate landmarks
(lakes, towers, stadiums, towns, ...) from OpenStreetMap along our route,
and save a clean table for the next notebook to build features on.

Steps:
1. Look up departure/destination coordinates (OurAirports)
2. Compute the direct route and a search corridor around it
3. Query OpenStreetMap (Overpass API) for candidate landmarks in that corridor
4. Enforce the *real* corridor width using great-circle cross-track distance
5. Clean up and save `data/processed/candidates_c81_kdlh.csv`
6. Sanity-check the result on a map


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
from vfr import airports, geo, osm

pd.set_option("display.max_columns", None)


## Step 1 — Airport coordinates

`vfr.airports.get_airport` downloads (and caches) OurAirports' free
`airports.csv` and looks up an identifier by either its `ident` or
`local_code` column — needed because `C81` is an FAA local identifier,
not an ICAO code.


In [2]:
dep = airports.get_airport("C81")
dest = airports.get_airport("KDLH")

route_distance_nm = geo.distance_nm(dep["lat"], dep["lon"], dest["lat"], dest["lon"])
route_bearing_deg = geo.bearing_deg(dep["lat"], dep["lon"], dest["lat"], dest["lon"])

print(dep)
print(dest)
print(f"Direct distance: {route_distance_nm:.1f} nm, initial bearing: {route_bearing_deg:.0f} deg")


{'ident': 'KC81', 'name': 'Campbell Airport', 'lat': 42.32460021972656, 'lon': -88.0740966796875, 'municipality': 'Grayslake', 'region': 'US-IL'}
{'ident': 'KDLH', 'name': 'Duluth International Airport', 'lat': 46.841873, 'lon': -92.198746, 'municipality': 'Duluth', 'region': 'US-MN'}
Direct distance: 323.4 nm, initial bearing: 328 deg


## Step 2 — Search corridor

Waypoints need to sit essentially right on the direct C81→KDLH course, so
there are two bands:
- **Preferred: +/-0.25 nm** — practically on the line
- **Fallback: +/-1 nm** — only used later (Notebook 07) to fill gaps where
  the 0.25 nm band has nothing for a long stretch (rural/forest terrain is
  sparse in OSM lakes/towers, and the tight band alone leaves some gaps
  of 20-40 nm on this route)

This notebook queries out to the *fallback* width (2 nm) so both bands
exist in the data, and tags each candidate with which band it falls in.
The actual "only widen where needed" logic is a route-assembly decision
that belongs in Notebook 07, once we have scores to decide what counts as
"needed" — here we're just collecting the candidate pool.

We don't yet know the exact great-circle corridor shape, so we start with
a generous rectangular bounding box around the two endpoints (padded by
the corridor half-width) and query OpenStreetMap within that box. The
*real* corridor constraint (perpendicular distance from the direct route)
gets enforced precisely in Step 4 using `geo.cross_track_distance_nm`.


In [3]:
PREFERRED_HALF_WIDTH_NM = 0.25  # the "essentially on the line" band
FALLBACK_HALF_WIDTH_NM = 1.0  # outer bound we collect data for, to fill gaps later
BBOX_PAD_NM = FALLBACK_HALF_WIDTH_NM + 2  # a little extra slack for the bounding box itself

route_start = (dep["lat"], dep["lon"])
route_end = (dest["lat"], dest["lon"])
bbox = geo.corridor_bbox(route_start, route_end, BBOX_PAD_NM)
bbox


(42.274600219726565, -92.248746, 46.891873, -88.0240966796875)

## Step 3 — Query OpenStreetMap

`vfr.osm.query_overpass` hits the public Overpass API for a handful of
landmark categories (lakes/ponds, reservoirs, water towers, other towers,
stadiums, quarries, towns/cities) inside the bounding box above, and
`parse_overpass_response` turns the result into a DataFrame.

This can take 10-60 seconds, and the public Overpass server occasionally
times out under load — `query_overpass` retries a few times automatically.


In [4]:
raw = osm.query_overpass(bbox)
raw_df = osm.parse_overpass_response(raw)
print(raw_df.shape)
raw_df.head()


(65101, 8)


,osm_id,osm_type,category,name,lat,lon,bbox_area_m2,tags
0,19188464,node,town,Duluth,46.783829,-92.105268,0.0,"{'capital': '6', 'name': 'Duluth', 'name:en': ..."
1,29885166,node,town,Woodstock,42.314753,-88.447430,0.0,"{'capital': '6', 'name': 'Woodstock', 'place':..."
2,29941173,node,town,Janesville,42.682977,-89.022679,0.0,"{'capital': '6', 'name': 'Janesville', 'name:a..."
3,29941752,node,town,Madison,43.074690,-89.384166,0.0,"{'alt_name:ru': 'Мэдисон', 'capital': '4', 'na..."
4,30080613,node,town,Sun Prairie,43.183278,-89.212830,0.0,"{'name': 'Sun Prairie', 'name:en': 'Sun Prairi..."


## Step 3b — Rivers, railroads, intersections, wind farms, and FAA navaids/obstacles

None of these are single-point OSM features the way lakes/towers/stadiums
are, so they get their own resolvers in `vfr.osm` rather than an entry in
`CANDIDATE_SPECS`:
- **Rivers/railroads**: resolved to the point where the route crosses
  them (a pilotage checkpoint for either is "the point you cross it," not
  the feature as a whole).
- **Highway intersections**: not a taggable OSM feature at all -- a node
  shared by 2+ genuinely different *numbered* roads (major highways only:
  motorway/trunk/primary/secondary).
- **Wind farms**: individually-mapped turbines, clustered into one
  farm-level candidate per cluster.

VOR navaids and towers/obstacles come from the FAA's own NASR/Digital
Obstacle File data (`vfr.faa_data`) instead of OSM tags -- the FAA is the
authority on its own navaid network, and DOF gives real obstacle
height/lighting instead of a generic `man_made=tower` OSM tag (filtered
to >=200ft AGL, the FAA/Part 77 general obstruction-notification
threshold). That data is downloaded once and cached under
`data/raw/faa_nasr/` (28-day publication cycle, much coarser than a
per-route query, so it isn't worth re-fetching every run).

In [5]:
RIVER_FILTER = '["waterway"="river"]["intermittent"!="yes"]'
RAILROAD_FILTER = '["railway"="rail"]["service"!~".*"]'

river_ways = osm.query_line_features(bbox, RIVER_FILTER)
river_df = osm.find_line_crossings(river_ways, route_start, route_end)
river_df["category"] = "river"

rail_ways = osm.query_line_features(bbox, RAILROAD_FILTER)
rail_df = osm.find_line_crossings(rail_ways, route_start, route_end)
rail_df["category"] = "railroad"

highway_ways, node_coords = osm.query_major_highways(bbox)
intersection_df = osm.find_intersections(highway_ways, node_coords)

turbine_df = osm.query_wind_turbines(bbox)
windfarm_df = osm.find_wind_farms(turbine_df)

raw_df = pd.concat([raw_df, river_df, rail_df, intersection_df, windfarm_df], ignore_index=True)
print(
    f"+{len(river_df)} river crossings, +{len(rail_df)} railroad crossings, "
    f"+{len(intersection_df)} intersections, +{len(windfarm_df)} wind farms"
)
raw_df["category"].value_counts()

+46 river crossings, +22 railroad crossings, +3005 intersections, +48 wind farms


category
lake_or_pond    62468
intersection     3005
quarry           1529
water_tower       830
town              190
stadium            78
wind_farm          48
river              46
railroad           22
reservoir           6
Name: count, dtype: int64

In [6]:
from vfr import faa_data

FAA_CACHE_DIR = PROJECT_ROOT / "data" / "raw" / "faa_nasr"
nav_csv_path, apt_csv_path, dof_dat_path = faa_data.ensure_nasr_data(FAA_CACHE_DIR)

vor_df = faa_data.load_vor_navaids(nav_csv_path, bbox)
obstacle_df = faa_data.load_obstacles(dof_dat_path, bbox, min_agl_ft=200)

raw_df = pd.concat([raw_df, vor_df, obstacle_df], ignore_index=True)
print(f"+{len(vor_df)} VOR navaids, +{len(obstacle_df)} FAA obstacles (>=200ft AGL)")
raw_df["category"].value_counts()

+13 VOR navaids, +3365 FAA obstacles (>=200ft AGL)


category
lake_or_pond    62468
tower            3365
intersection     3005
quarry           1529
water_tower       830
town              190
stadium            78
wind_farm          48
river              46
railroad           22
vor                13
reservoir           6
Name: count, dtype: int64

## Step 4 — Enforce the real corridor

For every candidate, compute:
- `cross_track_nm`: perpendicular distance from the direct C81→KDLH course
- `along_track_nm`: distance along the course from C81 to the candidate's
  projection onto the route (this is what Notebook 07 will use to space
  waypoints out)

Keep candidates within the *fallback* band (+/-1 nm) and not too far
behind the departure or past the destination, and flag which ones also
fall inside the tighter *preferred* band (+/-0.25 nm) with
`within_preferred_corridor`.


In [7]:
def add_route_distances(df):
    df = df.copy()
    df["cross_track_nm"] = df.apply(
        lambda r: geo.cross_track_distance_nm(r["lat"], r["lon"], route_start, route_end), axis=1
    )
    df["along_track_nm"] = df.apply(
        lambda r: geo.along_track_distance_nm(r["lat"], r["lon"], route_start, route_end), axis=1
    )
    return df

candidates_df = add_route_distances(raw_df)

MARGIN_NM = 5
in_corridor = (
    candidates_df["cross_track_nm"].abs() <= FALLBACK_HALF_WIDTH_NM
) & (
    candidates_df["along_track_nm"].between(-MARGIN_NM, route_distance_nm + MARGIN_NM)
)
candidates_df = candidates_df[in_corridor].reset_index(drop=True)
candidates_df["within_preferred_corridor"] = (
    candidates_df["cross_track_nm"].abs() <= PREFERRED_HALF_WIDTH_NM
)
print(candidates_df.shape)
print(f"Within preferred +/-{PREFERRED_HALF_WIDTH_NM} nm band: {candidates_df['within_preferred_corridor'].sum()}")
candidates_df.sort_values("along_track_nm").head(10)


(997, 11)
Within preferred +/-0.25 nm band: 288


,osm_id,osm_type,category,name,lat,lon,bbox_area_m2,tags,cross_track_nm,along_track_nm,within_preferred_corridor
357,411077224,way,lake_or_pond,NaN,42.317178,-88.090520,7.420565e+04,"{'natural': 'water', 'water': 'pond'}",-0.854479,0.002675,False
371,455176562,way,lake_or_pond,NaN,42.320756,-88.087914,9.302507e+03,"{'natural': 'water', 'water': 'pond'}",-0.643324,0.124935,False
346,318899793,way,lake_or_pond,NaN,42.326734,-88.078335,1.921834e+03,"{'natural': 'water', 'water': 'pond'}",-0.093074,0.207700,True
370,455176561,way,lake_or_pond,NaN,42.319992,-88.093310,2.589483e+04,"{'natural': 'water', 'water': 'pond'}",-0.871380,0.211464,False
391,497565089,way,lake_or_pond,NaN,42.323286,-88.094383,1.024706e+04,"{'natural': 'water', 'water': 'pond'}",-0.808262,0.404830,False
295,193047204,way,lake_or_pond,NaN,42.318090,-88.070942,3.771763e+03,"{'natural': 'water', 'water': 'pond'}",-0.085556,0.406285,True
359,413779302,way,lake_or_pond,NaN,42.326042,-88.052668,1.541100e+04,"{'natural': 'water', 'water': 'pond'}",0.855524,0.424682,False
294,193047203,way,lake_or_pond,NaN,42.317124,-88.070864,8.521167e+03,"{'natural': 'water', 'water': 'pond'}",-0.113005,0.457487,True
296,193047205,way,lake_or_pond,NaN,42.318761,-88.067231,1.777582e+04,"{'natural': 'water', 'water': 'pond'}",0.075853,0.458338,True
745,3319170,relation,lake_or_pond,NaN,42.313631,-88.076190,1.045336e+06,"{'natural': 'water', 'type': 'multipolygon', '...",-0.424307,0.512219,False


## Step 5 — Clean up

Drop exact-duplicate coordinates (the same feature sometimes comes back
as both a way and an overlapping relation), and take a look at what
categories we actually found — this is the sanity check for whether the
Overpass query and corridor filter are doing something reasonable before
we build features on top of it.


In [8]:
candidates_df = candidates_df.drop_duplicates(subset=["lat", "lon"]).reset_index(drop=True)
candidates_df["category"].value_counts()


category
lake_or_pond    736
tower            83
intersection     75
river            46
railroad         22
water_tower      17
quarry           10
town              4
stadium           2
wind_farm         1
Name: count, dtype: int64

## Step 5b — Cut obvious noise

OpenStreetMap tags an enormous number of farm ponds and drainage features
with the exact same `natural=water` tag as an actual lake — in this
corridor that's ~10,000 water polygons, and 91% of them have no name at
all. A pond that small and anonymous isn't a "big visual waypoint," so
drop unnamed water/reservoir features below a rough size threshold.

`bbox_area_m2` (computed in `vfr.osm` from each way/relation's bounding
box) is a cheap *overestimate* of true polygon area — good enough to
separate real lakes from farm ponds. Notebook 02 will compute a tighter
area feature from full geometry for whatever survives this cut.


In [9]:
MIN_UNNAMED_WATER_AREA_M2 = 40_000  # ~10 acres

is_small_unnamed_water = (
    candidates_df["category"].isin(["lake_or_pond", "reservoir"])
    & candidates_df["name"].isna()
    & (candidates_df["bbox_area_m2"] < MIN_UNNAMED_WATER_AREA_M2)
)
print(f"Dropping {is_small_unnamed_water.sum()} small/unnamed water features")

candidates_df = candidates_df[~is_small_unnamed_water].reset_index(drop=True)
candidates_df["category"].value_counts()


Dropping 570 small/unnamed water features


category
lake_or_pond    166
tower            83
intersection     75
river            46
railroad         22
water_tower      17
quarry           10
town              4
stadium           2
wind_farm         1
Name: count, dtype: int64

## Step 6 — Save

Save the cleaned candidate table for Notebook 02 (feature engineering).
The `tags` column (a dict of raw OSM tags) is serialized to a JSON string
so it survives a round-trip through CSV.


In [10]:
import json

OUT_PATH = PROJECT_ROOT / "data" / "processed" / "candidates_c81_kdlh.csv"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

out_df = candidates_df.copy()
out_df["tags"] = out_df["tags"].apply(json.dumps)
out_df.to_csv(OUT_PATH, index=False)

n_preferred = int(out_df["within_preferred_corridor"].sum())
print(f"Saved {len(out_df)} candidates to {OUT_PATH} ({n_preferred} within the preferred +/-{PREFERRED_HALF_WIDTH_NM} nm band)")
out_df.head()


Saved 426 candidates to /Users/asdaq/Desktop/Projects/vfr_route/data/processed/candidates_c81_kdlh.csv (150 within the preferred +/-0.25 nm band)


,osm_id,osm_type,category,name,lat,lon,bbox_area_m2,tags,cross_track_nm,along_track_nm,within_preferred_corridor
0,151331909,node,town,Superior,46.720774,-92.104080,0.0,"{""capital"": ""6"", ""ele"": ""196"", ""gnis:feature_i...",-0.906271,315.222733,False
1,153420031,node,town,Montello,43.791861,-89.328819,0.0,"{""capital"": ""6"", ""ele"": ""242"", ""gnis:feature_i...",0.062879,103.874318,True
2,153546173,node,town,Round Lake,42.353355,-88.093414,0.0,"{""ele"": ""243"", ""gnis:feature_id"": ""416994"", ""n...",0.174845,1.919655,True
3,153566547,node,town,Round Lake Beach,42.371688,-88.090081,0.0,"{""ele"": ""233"", ""gnis:feature_id"": ""416995"", ""n...",0.877854,2.779455,False
4,353870611,node,lake_or_pond,Big Falls Flowage,45.555885,-90.960424,0.0,"{""ele"": ""372"", ""gnis:feature_id"": ""1561722"", ""...",-0.602629,230.635294,False


## Step 7 — Map sanity check

A quick visual gut check: does the route line look right, and do the
candidate markers look like plausible landmarks along it? Worth eyeballing
a few on a real map (e.g. search the name on Google Maps) before trusting
this data in the next notebook.


In [11]:
import folium
from vfr.config import FAA_VFR_SECTIONAL_URL, VFR_SECTIONAL_MAX_ZOOM, VFR_SECTIONAL_MIN_ZOOM

mid_lat = (dep["lat"] + dest["lat"]) / 2
mid_lon = (dep["lon"] + dest["lon"]) / 2
# VFR Sectional is the default base layer (tiles=None below, then added
# first) since that's the chart a pilot actually plans against;
# OpenStreetMap is added second as a togglable alternate -- switch to it
# via the layer control (top right) for whole-route context. zoom_start
# is pinned to the sectional's own min zoom (8) rather than a wider
# whole-route zoom, since the tile service renders nothing below that --
# starting any wider would just show a blank chart on load.
# minZoom=6 is set explicitly on the map itself so you can still scroll
# out past the sectional layer's own min zoom of 8 to see the whole route
# -- the chart just goes blank below 8, but the polyline/markers are
# vector overlays and stay visible regardless. NOTE: folium.Map's own
# min_zoom= kwarg only takes effect when tiles= is a plain string (it
# gets forwarded into folium's *implicit* default TileLayer); since we
# pass tiles=None and add layers manually below, that kwarg is silently
# dropped -- minZoom (Leaflet's actual camelCase option name, passed
# through Map's **kwargs straight into the JS map options) is what
# actually reaches the Leaflet map object.
m = folium.Map(
    location=[mid_lat, mid_lon],
    zoom_start=VFR_SECTIONAL_MIN_ZOOM,
    minZoom=6,
    tiles=None,
    height="100%",
)
# height="100%" so the saved standalone HTML fills the whole browser tab --
# this is meant to be opened as its own page, not viewed inline in the
# notebook cell output (a plain "100%" collapses to 0 there since VS Code's
# notebook output area doesn't give the map a real height to fill against)
folium.TileLayer(
    tiles=FAA_VFR_SECTIONAL_URL,
    attr="FAA Aeronautical Information Services",
    name="VFR Sectional",
    max_zoom=VFR_SECTIONAL_MAX_ZOOM,
    min_zoom=VFR_SECTIONAL_MIN_ZOOM,
).add_to(m)
folium.TileLayer(tiles="OpenStreetMap", name="OpenStreetMap", show=False).add_to(m)
folium.LayerControl().add_to(m)

folium.PolyLine([[dep["lat"], dep["lon"]], [dest["lat"], dest["lon"]]], color="blue", weight=2).add_to(m)
folium.Marker([dep["lat"], dep["lon"]], tooltip=f"Departure: {dep['ident']}", icon=folium.Icon(color="green")).add_to(m)
folium.Marker([dest["lat"], dest["lon"]], tooltip=f"Destination: {dest['ident']}", icon=folium.Icon(color="red")).add_to(m)

for _, r in candidates_df.iterrows():
    color = "orange" if r["within_preferred_corridor"] else "gray"
    folium.CircleMarker(
        [r["lat"], r["lon"]],
        radius=4,
        popup=f"{r['name'] or '(unnamed)'} ({r['category']}, cross-track {r['cross_track_nm']:.2f} nm)",
        color=color,
        fill=True,
    ).add_to(m)

map_path = PROJECT_ROOT / "data" / "processed" / "candidates_map.html"
m.save(str(map_path))
print(f"Map saved to {map_path}")
m

Map saved to /Users/asdaq/Desktop/Projects/vfr_route/data/processed/candidates_map.html
